# Notebook 01 — Exploration des données

Ce notebook charge la fixture synthétique, inspecte la distribution des événements,
et valide le pipeline de normalisation bout-en-bout.

> **Données** : `tests/integration/fixtures/sample_logs.csv` — 103 événements synthétiques,
> 7 utilisateurs, scénario de password spray (T1110.003) le 16 mai 2026 à 14h.

In [ ]:
import sys
sys.path.insert(0, '../src')

import csv
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Chargement de la fixture

In [ ]:
FIXTURE = Path('../tests/integration/fixtures/sample_logs.csv')
df = pd.read_csv(FIXTURE, parse_dates=['@timestamp'])
print(f'Lignes : {len(df)} | Colonnes : {list(df.columns)}')
df.head()

## 2. Distribution des Event IDs

In [ ]:
event_counts = df['data.win.system.eventID'].value_counts()
print(event_counts)

fig, ax = plt.subplots(figsize=(7, 4))
event_counts.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Distribution des Event IDs Windows')
ax.set_xlabel('Event ID')
ax.set_ylabel('Nombre d\'occurrences')
plt.tight_layout()
plt.show()

## 3. Activité par utilisateur et par jour

In [ ]:
df['date'] = df['@timestamp'].dt.date
df['user'] = df['data.win.eventdata.targetUserName'].fillna(
    df['data.win.eventdata.subjectUserName']
)

pivot = df.groupby(['user', 'date']).size().unstack(fill_value=0)
print(pivot)

fig, ax = plt.subplots(figsize=(9, 5))
pivot.plot(kind='bar', ax=ax)
ax.set_title('Nombre d\'événements par utilisateur et par jour')
ax.set_xlabel('Utilisateur')
ax.set_ylabel('Événements')
ax.legend(title='Date')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 4. Identification du scénario de Password Spray

In [ ]:
failed = df[df['data.win.system.eventID'] == 4625].copy()
failed['hour'] = failed['@timestamp'].dt.hour
failed['minute'] = failed['@timestamp'].dt.minute

print(f'Total échecs de connexion (4625) : {len(failed)}')
print(f'Utilisateurs ciblés : {sorted(failed["user"].dropna().unique())}')
print(f'IP sources : {failed["data.win.eventdata.ipAddress"].dropna().unique()}')
print('\nTimeline des échecs :')
print(failed[['@timestamp', 'user', 'data.win.eventdata.ipAddress']].sort_values('@timestamp').to_string())

## 5. Normalisation via WazuhAdapter

In [ ]:
from ueba.adapters.wazuh import WazuhAdapter

with FIXTURE.open(newline='', encoding='utf-8') as f:
    records = list(csv.DictReader(f))

adapter = WazuhAdapter()
events = adapter.normalize(records)

print(f'Événements normalisés : {len(events)}')
print(f'Utilisateurs : {sorted({e.user for e in events})}')
print(f'\nExemple :'); print(events[0])